# EXP-105 · InternVL3-8B LoRA 파인튜닝 (A100 전용)

Qwen과 계열이 다른 모델로 앙상블 다양성 확보. 레시피는 EXP-102와 동일(정답 토큰 loss, 선지 섞기, dev 추가, valid 누수 차단).

**순서**: 모델 로드 → **제로샷 100문항 점검(80% 이상이면 정상)** → LoRA 학습 → valid → test

**기존 3B 학습 코드 대비 개선점**
1. **정답 토큰에만 loss** (기존: `labels = input_ids.clone()` → 질문·이미지 토큰까지 학습)
2. **선지 순서 무작위 섞기 증강** → 위치 편향 제거, "거의 같은 선지" 구별 학습
3. **dev pseudo-label 2,026개 추가** (교육생 응답 5개 기반)
4. **valid 672문항 누수 차단** (`train_clean_v2`에 EXP-007 valid 672 중 422개가 섞여 있었음 → 제외)
5. LoRA r=16, α=32 (α=2r), 체크포인트 드라이브 백업 + 재개

**GPU 자동 선택**: A100 → 8B bf16 / T4 → 4B fp16 (4bit 양자화는 T4에서 30배 느려서 사용 안 함)

**예상 시간 (A100·8B)**: 학습 1.5~2.5시간 → valid 5분 → test 40분

In [ ]:
!pip install -q -U transformers accelerate peft bitsandbytes
!apt-get install -y -qq p7zip-full > /dev/null
import transformers, peft, torch
print(transformers.__version__, peft.__version__, torch.cuda.get_device_name(0))
!pip uninstall -y torchao

In [ ]:
import os, glob, shutil, subprocess
from google.colab import drive
drive.mount("/content/drive")
cnt = lambda d: len(glob.glob(f"/content/data/{d}/*.jpg"))
print("현재 이미지 수 → train:", cnt("train"), "dev:", cnt("dev"), "test:", cnt("test"))

if cnt("train") < 6714 or cnt("dev") < 2000 or cnt("test") < 6714:
    ZIP_SRC = glob.glob("/content/drive/MyDrive/ssafy/ai-challenge/ssafy-16-2-ai*.zip")[0]
    ZIP_LOCAL = "/content/data.zip"
    if not os.path.exists(ZIP_LOCAL) or os.path.getsize(ZIP_LOCAL) != os.path.getsize(ZIP_SRC):
        print("zip 복사 중..."); shutil.copy(ZIP_SRC, ZIP_LOCAL)
    shutil.rmtree("/content/data", ignore_errors=True); os.makedirs("/content/data")
    r = subprocess.run(["7z","x","-y","-pocrisnotenough","-o/content/data",ZIP_LOCAL], capture_output=True, text=True)
    print("종료 코드:", r.returncode, r.stderr[-800:])
    print("다시 센 이미지 수 → train:", cnt("train"), "dev:", cnt("dev"), "test:", cnt("test"))

In [ ]:
import os, glob, subprocess, time, random, math
import pandas as pd, numpy as np
from google.colab import drive
drive.mount("/content/drive")

SHARE_DIR = "/content/drive/MyDrive/ssafy/ai-challenge"
EXP_DIR, SUB_DIR = f"{SHARE_DIR}/experiment_data", f"{SHARE_DIR}/submissions"
ZIP_PATH = glob.glob(f"{SHARE_DIR}/ssafy-16-2-ai*.zip")[0]
DATA_DIR = "/content/data"
if not glob.glob(f"{DATA_DIR}/**/test.csv", recursive=True):
    os.makedirs(DATA_DIR, exist_ok=True)
    subprocess.run(["7z","x","-y","-pocrisnotenough",f"-o{DATA_DIR}",ZIP_PATH],check=True,stdout=subprocess.DEVNULL)
DATA_ROOT = os.path.dirname(glob.glob(f"{DATA_DIR}/**/test.csv", recursive=True)[0])
resolve = lambda p: p if os.path.isabs(str(p)) else os.path.join(DATA_ROOT, str(p))
COLS = ["id","path","question","a","b","c","d","answer"]

# valid = EXP-007과 동일한 672문항 (앙상블 가중치 결정용 → 절대 학습에 넣지 않음)
valid_df = pd.read_csv(f"{EXP_DIR}/EXP-007_train6042_resolution1280_valid_seed43.csv")[COLS]
VALID_IDS = set(valid_df.id)

# train = 정제본(clean_v2 train+valid) 에서 valid 672 제외  (정제 라벨 사용)
clean = pd.concat([pd.read_csv(f"{SHARE_DIR}/split/train_clean_v2_seed43.csv"),
                   pd.read_csv(f"{SHARE_DIR}/split/valid_clean_v2_seed43.csv")])[COLS]
train_df = clean[~clean.id.isin(VALID_IDS)]
dev_df = pd.read_csv(f"{SHARE_DIR}/dev_pseudo_all_2026.csv")[COLS]
USE_DEV = True
full_train = pd.concat([train_df, dev_df]) if USE_DEV else train_df
full_train = full_train.sample(frac=1, random_state=43).reset_index(drop=True)

test_df = pd.read_csv(f"{DATA_ROOT}/test.csv")
for d in (valid_df, full_train, test_df): d["path"] = d["path"].map(resolve)
assert not set(full_train.id) & VALID_IDS
print("train(clean, valid 제외):", len(train_df), "+ dev:", len(dev_df) if USE_DEV else 0, "=", len(full_train))
print("valid:", len(valid_df), "test:", len(test_df))
print("누락 이미지:", (~full_train.path.map(os.path.exists)).sum())

In [ ]:
# ===== 모델 로드 =====
from transformers import AutoProcessor, AutoModelForImageTextToText
from peft import LoraConfig, get_peft_model, PeftModel
from PIL import Image
import torch.nn.functional as F
Image.MAX_IMAGE_PIXELS = None

MODEL_ID = "OpenGVLab/InternVL3-8B-hf"      # transformers 기본 지원 버전(-hf)
TAG = "EXP-105_internvl3-8b_lora_r16"
CKPT_DIR = f"{SHARE_DIR}/saved_models/{TAG}"
MAX_PATCHES = 6                   # 448px 타일 최대 개수(+썸네일 1). 토큰 = 타일 수 x 256
TRAIN_MAX_PIXELS = INFER_MAX_PIXELS = 1280*32*32   # 원본이 너무 크면 먼저 축소
SEED = 43; random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
DTYPE = torch.bfloat16
assert "A100" in torch.cuda.get_device_name(0) or "H100" in torch.cuda.get_device_name(0), "A100에서 실행하세요"

model = AutoModelForImageTextToText.from_pretrained(MODEL_ID, dtype=DTYPE, device_map="cuda", attn_implementation="sdpa")
processor = AutoProcessor.from_pretrained(MODEL_ID)
tok = processor.tokenizer
ip = processor.image_processor
for k,v in [("max_patches",MAX_PATCHES),("crop_to_patches",True)]:
    if hasattr(ip,k): setattr(ip,k,v)
print(type(model).__name__, "| max_patches:", getattr(ip,"max_patches",None))

In [ ]:
# ===== 프롬프트 · 선지 섞기 · 정답 토큰에만 loss =====
def fit(img, maxp):
    w,h = img.size
    if w*h > maxp:
        s=(maxp/(w*h))**0.5; img=img.resize((max(32,int(w*s)),max(32,int(h*s))), Image.BICUBIC)
    return img

def prompt(q, opts):
    return (f"이미지를 자세히 보고 질문에 답하세요. 이미지 속 글자를 정확히 읽는 것이 중요합니다.\n\n"
            f"질문: {q}\n" + "".join(f"{l}. {o}\n" for l,o in zip("abcd",opts)) +
            "\n정답 선지의 알파벳(a, b, c, d) 한 글자만 출력하세요.")

def build_train_example(r, shuffle=True):
    opts = [str(r[c]) for c in "abcd"]; gold = "abcd".index(str(r["answer"]).strip().lower())
    order = list(range(4))
    if shuffle: random.shuffle(order)                     # 선지 순서 섞기 증강
    new_opts = [opts[i] for i in order]; new_gold = "abcd"[order.index(gold)]
    img = fit(Image.open(r["path"]).convert("RGB"), TRAIN_MAX_PIXELS)
    msgs = [{"role":"user","content":[{"type":"image"},{"type":"text","text":prompt(r["question"], new_opts)}]}]
    prefix = processor.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)
    target = new_gold + "<|im_end|>"
    enc = processor(text=[prefix + target], images=[img], return_tensors="pt")
    n_tgt = len(tok.encode(target, add_special_tokens=False))
    labels = torch.full_like(enc["input_ids"], -100)
    labels[0, -n_tgt:] = enc["input_ids"][0, -n_tgt:]     # 정답 글자 + 종료 토큰에만 loss
    enc["labels"] = labels
    return enc



In [ ]:
# ===== 제로샷 점검 (100문항, 약 2분) — 파이프라인이 정상인지 확인 =====
model.eval()
CHOICE_IDS = []
for ch in "abcd":
    ss=set()
    for x in [ch, ch.upper(), " "+ch, " "+ch.upper()]:
        t=tok.encode(x, add_special_tokens=False)
        if len(t)==1: ss.add(t[0])
    CHOICE_IDS.append(sorted(ss))
print("선지 토큰:", CHOICE_IDS)

@torch.inference_mode()
def score(r):
    img = fit(Image.open(r["path"]).convert("RGB"), INFER_MAX_PIXELS)
    msgs=[{"role":"user","content":[{"type":"image"},{"type":"text","text":prompt(r["question"], [str(r[c]) for c in "abcd"])}]}]
    text=processor.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)
    inp=processor(text=[text], images=[img], return_tensors="pt").to("cuda")
    with torch.autocast("cuda", dtype=DTYPE):
        try: lg=model(**inp, logits_to_keep=1, use_cache=False).logits[0,-1].float()
        except TypeError: lg=model(**inp, use_cache=False).logits[0,-1].float()
    return F.softmax(torch.stack([torch.logsumexp(lg[ids],0) for ids in CHOICE_IDS]),0).cpu().numpy()

t0=time.time(); sub=valid_df.head(100)
pz=np.array([score(r) for _,r in sub.iterrows()])
acc=(pz.argmax(1)==sub.answer.map(dict(zip("abcd",range(4)))).values).mean()
print(f"제로샷 100문항: {acc:.2f} | {(time.time()-t0)/100:.2f}s/문항 | 예측분포 {np.bincount(pz.argmax(1),minlength=4)}")
e = build_train_example(full_train.iloc[0])
print("라벨 토큰:", repr(tok.decode(e["labels"][0][e["labels"][0]!=-100])), "| 시퀀스 길이:", e["input_ids"].shape[1])
print("→ 제로샷 0.80 이상, 라벨 토큰이 'x<|im_end|>' 형태면 학습 진행")

In [ ]:
# ===== LoRA (언어 모델 부분에만) =====
model.gradient_checkpointing_enable(gradient_checkpointing_kwargs={"use_reentrant": False})
model.enable_input_require_grads()
model.config.use_cache = False
lora = LoraConfig(r=16, lora_alpha=32, lora_dropout=0.05, bias="none", task_type="CAUSAL_LM",
                  target_modules=r".*language_model.*\.(q_proj|k_proj|v_proj|o_proj|gate_proj|up_proj|down_proj)")
START_STEP = 0
latest = sorted([p for p in glob.glob(f"{CKPT_DIR}/step_*") if os.path.exists(f"{p}/adapter_model.safetensors")],
                key=lambda p:int(p.split("_")[-1]))
if latest:
    model = PeftModel.from_pretrained(model, latest[-1], is_trainable=True)
    START_STEP = int(latest[-1].split("_")[-1]); print("재개:", latest[-1])
else:
    model = get_peft_model(model, lora)
for p in model.parameters():
    if p.requires_grad: p.data = p.data.float()
model.print_trainable_parameters()

## 학습 (끊기면 모델 셀부터 다시 실행 → 마지막 체크포인트에서 자동 재개)

In [ ]:
EPOCHS, GRAD_ACCUM, LR = 1, 8, 1e-4
SAVE_EVERY = 100   # optimizer step 기준
n_updates = math.ceil(len(full_train)*EPOCHS/GRAD_ACCUM)
params = [p for p in model.parameters() if p.requires_grad]
opt = torch.optim.AdamW(params, lr=LR, weight_decay=0.0)
from transformers import get_cosine_schedule_with_warmup
sched = get_cosine_schedule_with_warmup(opt, int(n_updates*0.03), n_updates)
for _ in range(START_STEP): sched.step()
scaler = torch.amp.GradScaler(enabled=(DTYPE==torch.float16))

model.train(); step = START_STEP; run_loss = 0; t0 = time.time()
start_i = START_STEP*GRAD_ACCUM
order = list(range(len(full_train)))*EPOCHS
print(f"총 {n_updates} step, {start_i}번째 샘플부터 시작")
for i in range(start_i, len(order)):
    random.seed(SEED+i)   # 재개해도 같은 섞기
    try:
        b = {k:v.to("cuda") for k,v in build_train_example(full_train.iloc[order[i]]).items()}
        with torch.autocast("cuda", dtype=DTYPE):
            loss = model(**b).loss / GRAD_ACCUM
        scaler.scale(loss).backward(); run_loss += loss.item()
    except torch.cuda.OutOfMemoryError:
        opt.zero_grad(set_to_none=True); torch.cuda.empty_cache(); print("OOM skip", i); continue
    if (i+1) % GRAD_ACCUM == 0:
        scaler.unscale_(opt); torch.nn.utils.clip_grad_norm_(params, 1.0)
        scaler.step(opt); scaler.update(); opt.zero_grad(set_to_none=True); sched.step(); step += 1
        if step % 10 == 0:
            el=time.time()-t0; done=i+1-start_i
            print(f"step {step}/{n_updates} loss {run_loss/10:.4f} | {el/done:.2f}s/샘플, 남은 {el/done*(len(order)-i-1)/60:.0f}분")
            run_loss = 0
        if step % SAVE_EVERY == 0 or step == n_updates:
            model.save_pretrained(f"{CKPT_DIR}/step_{step}"); print("저장", step)
model.save_pretrained(f"{CKPT_DIR}/final"); print("학습 완료")

## valid 평가 → 기존 앙상블과 결합

In [ ]:
model.eval()
P = ["prob_a","prob_b","prob_c","prob_d"]
def run(df, out_csv, every=100):
    done = pd.read_csv(out_csv) if os.path.exists(out_csv) else pd.DataFrame(columns=["id"]+P)
    todo = df[~df.id.astype(str).isin(set(done.id.astype(str)))]; rows=[]; t0=time.time()
    for n,(_,r) in enumerate(todo.iterrows(),1):
        rows.append([r.id,*score(r)])
        if n%every==0 or n==len(todo):
            done=pd.concat([done,pd.DataFrame(rows,columns=done.columns)],ignore_index=True); done.to_csv(out_csv,index=False); rows=[]
            el=time.time()-t0; print(f"{n}/{len(todo)} 남은 {el/n*(len(todo)-n)/60:.0f}분")
    return done.set_index("id")

vp = run(valid_df, f"{EXP_DIR}/{TAG}_valid_probs.csv", every=50)
ids = valid_df.id.values; ans = valid_df.answer.map(dict(zip("abcd",range(4)))).values
new = vp.loc[ids, P].values.astype(float)
def probs(p): return pd.read_csv(p).set_index("id")[P]
base = 0.52*probs(f"{EXP_DIR}/EXP-007_train6042_resolution1280_valid_seed43.csv").loc[ids].values \
     + 0.48*probs(f"{EXP_DIR}/EXP-034_qwen7b_language_only_train1600_resolution1280_valid_seed43.csv").loc[ids].values
print(f"★ {TAG} 단독 valid: {(new.argmax(1)==ans).mean():.4f}   기존 앙상블: {(base.argmax(1)==ans).mean():.4f}")
for w in np.arange(0,1.01,0.1):
    print(f"w_new={w:.1f}: {(((1-w)*base+w*new).argmax(1)==ans).mean():.4f}")

## test 추론 + 제출 파일

In [ ]:
tp = run(test_df, f"{EXP_DIR}/{TAG}_test_probs.csv")

In [ ]:
W_NEW = 0.6   # ← valid 표 보고 결정
t_ids = test_df.id.values
tbase = 0.52*probs(f"{EXP_DIR}/EXP-007_train6042_resolution1280_test_probabilities_seed43.csv").loc[t_ids].values \
      + 0.48*probs(f"{EXP_DIR}/EXP-040_qwen7b_train1600_resolution1280_test_probabilities_seed43.csv").loc[t_ids].values
tnew = tp.loc[t_ids, P].values.astype(float)
for name, pr in [("single", tnew), (f"ens_w{int(W_NEW*10)}", (1-W_NEW)*tbase + W_NEW*tnew)]:
    sub = pd.DataFrame({"id": t_ids, "answer": ["abcd"[i] for i in pr.argmax(1)]})
    out = f"{SUB_DIR}/{TAG}_{name}.csv"; sub.to_csv(out, index=False); print("저장:", out, sub.answer.value_counts().to_dict())